# Hybrid Search RAG: Dense + Sparse Retrieval
This notebook demonstrates how to build a Hybrid Search pipeline from scratch using **BM25** (Sparse) and **Sentence Transformers** (Dense), fusing them with **Reciprocal Rank Fusion (RRF)**, and generating an answer with **Ollama**.

In [6]:
#!pip install rank_bm25 sentence-transformers scikit-learn numpy ollama

## 1. The Dataset
Let's create a sample dataset. Notice how some documents share exact keywords, while others share semantic meaning.

In [7]:
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming the tech industry.",
    "Machine learning allows computers to learn from data without explicit programming.",
    "A fast auburn canine leaps above an inactive hound.", # Semantic match for doc 0
    "Apple announced the new iPhone 15 today with a titanium body.",
    "You should eat an apple a day to keep the doctor away.", # Keyword match for 'apple'
    "Deep learning models require specialized hardware like GPUs to train efficiently."
]

query = "What is required to train deep neural networks?"

## 2. Sparse Retrieval (BM25)
BM25 looks for exact keyword matches. It's great for finding specific names, acronyms, or distinct terms.

In [8]:
from rank_bm25 import BM25Okapi

# Simple tokenizer: split by lowercase spaces
tokenized_corpus = [doc.lower().split(" ") for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

tokenized_query = query.lower().split(" ")
bm25_scores = bm25.get_scores(tokenized_query)

print("BM25 Scores:")
for i, score in enumerate(bm25_scores):
    print(f"Doc {i}: {score:.4f}")

BM25 Scores:
Doc 0: 0.0000
Doc 1: 1.6952
Doc 2: 0.2405
Doc 3: 0.0000
Doc 4: 0.0000
Doc 5: 0.2306
Doc 6: 3.0469


## 3. Dense Retrieval (Vector Search)
Vector search captures semantic meaning, even if exact keywords don't match.

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load a small, fast embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = embedder.encode(documents)
query_embedding = embedder.encode([query])

# Calculate cosine similarity between query and all docs
dense_scores = cosine_similarity(query_embedding, doc_embeddings)[0]

print("Dense Scores:")
for i, score in enumerate(dense_scores):
    print(f"Doc {i}: {score:.4f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1949.87it/s]


Dense Scores:
Doc 0: -0.0433
Doc 1: 0.2221
Doc 2: 0.3679
Doc 3: -0.0106
Doc 4: 0.0686
Doc 5: -0.0000
Doc 6: 0.6248


## 4. Reciprocal Rank Fusion (RRF)
Now we combine them. RRF doesn't look at the raw scores (because BM25 and Cosine Similarity are on different scales). Instead, it looks at the **Rank**.
Formula: `RRF Score = 1 / (k + Rank)` where `k` is a constant (usually 60).

In [10]:
def get_ranks(scores):
    # Returns a dictionary mapping doc_index to its rank (1 is best)
    # np.argsort sorts ascending, so we reverse it with [::-1]
    sorted_indices = np.argsort(scores)[::-1]
    ranks = {idx: rank + 1 for rank, idx in enumerate(sorted_indices)}
    return ranks

bm25_ranks = get_ranks(bm25_scores)
dense_ranks = get_ranks(dense_scores)

print("Ranks (Doc Index: Rank) - Lower rank is better")
print(f"BM25 Ranks:  {bm25_ranks}")
print(f"Dense Ranks: {dense_ranks}")

k = 60
rrf_scores = {}
for i in range(len(documents)):
    rrf_score = (1 / (k + bm25_ranks[i])) + (1 / (k + dense_ranks[i]))
    rrf_scores[i] = rrf_score

# Get top 2 documents based on RRF
top_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:2]

print("\n--- Top Retrieved Documents ---")
retrieved_context = ""
for idx in top_indices:
    print(f"[Doc {idx}] Score: {rrf_scores[idx]:.4f} -> {documents[idx]}")
    retrieved_context += documents[idx] + "\n"

Ranks (Doc Index: Rank) - Lower rank is better
BM25 Ranks:  {np.int64(6): 1, np.int64(1): 2, np.int64(2): 3, np.int64(5): 4, np.int64(4): 5, np.int64(3): 6, np.int64(0): 7}
Dense Ranks: {np.int64(6): 1, np.int64(2): 2, np.int64(1): 3, np.int64(4): 4, np.int64(5): 5, np.int64(3): 6, np.int64(0): 7}

--- Top Retrieved Documents ---
[Doc 6] Score: 0.0328 -> Deep learning models require specialized hardware like GPUs to train efficiently.
[Doc 1] Score: 0.0320 -> Artificial intelligence is transforming the tech industry.


## 5. Generation with Ollama
Finally, we pass the retrieved context to our local LLM to generate the answer. You mentioned `qwen2.5:7b-instruct` in your models file, so we'll use that.

In [11]:
import ollama

system_prompt = "You are a helpful assistant. Use the provided context to answer the user's question."
user_prompt = f"Context:\n{retrieved_context}\n\nQuestion: {query}"

print("Generating answer with qwen2.5:7b-instruct...")
response = ollama.chat(model='qwen2.5:7b-instruct', messages=[
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': user_prompt}
])

print("\n--- Answer ---")
print(response['message']['content'])

Generating answer with qwen2.5:7b-instruct...

--- Answer ---
To train deep neural networks, specialized hardware such as GPUs is required for efficient training.


Hybrid Search is currently the "gold standard," where systems combine the BM25 score with a Vector Search score to get the best of both worlds.